# Lab 11 · Routing, loops, reducers, Command and fan-out

**Time:** 50 min · **Work in teams of 3** · **You will:** make a graph **retry** when the model's label is invalid (a **loop with a limit**), see what a **reducer** does, use **`Command`** to update the state and choose the next node in one step, trigger the **recursion limit** on purpose, and build a **fan-out**: three branches that run **at the same time** and are combined by a **fan-in** node.

**Why it matters:** a real assistant must cope with a model that sometimes answers badly (retry, then give up politely), and must not do slow independent work one step at a time. These are the daily tools of a LangGraph workflow.

**The small model is in every graph here.** The router, the extraction and the final sentence are all Qwen2.5-3B on your GPU.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**. Predict before each run.*

In [ ]:
import sys, os, json, time, operator
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import check, DATA
from typing import TypedDict, Annotated, Literal
from lc_slm import SmallChatModel
from graph_viz import draw_graph
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.errors import GraphRecursionError, InvalidUpdateError
from langchain_core.messages import SystemMessage, HumanMessage
import pandas as pd

router_model = SmallChatModel(max_new_tokens=8)
talk_model = SmallChatModel(max_new_tokens=80)
json_model = SmallChatModel(max_new_tokens=60)
ROUTES = ["act", "ask", "chat", "other"]

## 11.1 A router that retries when the label is invalid
The model must answer with **one word** from a list. Sometimes it does not. The graph must **never trust the label until it has checked it**. If the label is not valid, the graph goes **back** to the classify node (a **loop**), with a stricter prompt, up to **3 attempts**, then it gives up with a **fallback**.

To make the retry happen reliably in this lab, the **first attempt uses a loose prompt** that does not tell the model to answer with one word.

In [ ]:
LOOSE = "You help a smart-home assistant. What does the user want?"
STRICT = ("You route messages for a smart-home assistant. Reply with exactly ONE word from this list and nothing else:\n"
          "act (do something in the home), ask (a question), chat (greeting or thanks), other (anything else).")

class RouteState(TypedDict, total=False):
    message: str
    route: str                                   # a valid label, or "" while the label is invalid
    attempts: int                                # how many times we asked
    log: Annotated[list, operator.add]           # a REDUCER: every update is appended, not overwritten
    reply: str

def classify(state: RouteState) -> dict:
    n = state.get("attempts", 0) + 1
    prompt = LOOSE if n == 1 else STRICT
    raw = router_model.invoke([SystemMessage(prompt), HumanMessage(state["message"])]).content.strip().lower()
    label = next((r for r in ROUTES if raw.startswith(r)), "")            # the code checks the label
    return {"route": label, "attempts": n, "log": [f"attempt {n}: model said {raw!r} -> {label or 'INVALID'}"]}

def after_classify(state: RouteState) -> str:
    """The conditional edge: valid label -> go on; invalid and attempts left -> loop back; else give up."""
    if state["route"]:
        return state["route"]
    return "classify" if state["attempts"] < 3 else "fallback"

def respond(kind):
    def node(state: RouteState) -> dict:
        return {"reply": f"[{kind} branch] handling: {state['message']}"}
    return node

g = StateGraph(RouteState)
g.add_node("classify", classify)
for kind in ROUTES:
    g.add_node(kind, respond(kind))
g.add_node("fallback", lambda s: {"reply": "Sorry, I did not understand. Could you rephrase?"})
g.add_edge(START, "classify")
g.add_conditional_edges("classify", after_classify, {**{k: k for k in ROUTES}, "classify": "classify", "fallback": "fallback"})
for kind in ROUTES + ["fallback"]:
    g.add_edge(kind, END)
router_app = g.compile()
draw_graph(router_app, "Router with a retry loop (dashed arrows are conditional)")

In [ ]:
out = router_app.invoke({"message": "Turn on the kitchen light"})
for line in out["log"]:
    print(line)
print("route:", out["route"], "| attempts:", out["attempts"], "| reply:", out["reply"])

**Read it:** the `log` shows what happened. The first attempt, with the loose prompt, probably gave a sentence, which is **not** a valid label, so the graph went **back**. The second attempt used the strict prompt. The `log` has several lines because of the **reducer** `Annotated[list, operator.add]`: without it each update would **overwrite** the last one.

In [ ]:
check(out["attempts"] >= 2 and out["route"] in ROUTES,
      f"The first answer was invalid, the graph looped back, and attempt {out['attempts']} gave the valid label '{out['route']}'. Retry is a conditional edge that points backwards.",
      "the loose prompt should have produced an invalid label first, so attempts should be 2 or more.")

### 🔧 Try it 11.1 — Your own message
**Predict first:** will your message need a retry? Which route will it take?

In [ ]:
my_message = "It is getting warm in here"                   # 🔧 your own message
out = router_app.invoke({"message": my_message})
print("\n".join(out["log"]))
print("->", out["route"], "|", out["reply"])

## 11.2 `Command`: update the state and choose the next node in one step
So far a node updated the state and a **separate** function (the conditional edge) chose the next node. With **`Command`** a node does both: `Command(update={...}, goto="node")`. Same behaviour, less wiring, and the decision sits next to the code that made it.

In [ ]:
class CmdState(TypedDict, total=False):
    message: str
    attempts: int
    log: Annotated[list, operator.add]
    reply: str

def classify_cmd(state: CmdState) -> Command[Literal["act", "ask", "chat", "other", "classify", "fallback"]]:
    n = state.get("attempts", 0) + 1
    raw = router_model.invoke([SystemMessage(LOOSE if n == 1 else STRICT), HumanMessage(state["message"])]).content.strip().lower()
    label = next((r for r in ROUTES if raw.startswith(r)), "")
    note = {"attempts": n, "log": [f"attempt {n}: {raw!r} -> {label or 'INVALID'}"]}
    if label:
        return Command(update=note, goto=label)                      # valid: write the log AND go to the branch
    return Command(update=note, goto="classify" if n < 3 else "fallback")

c = StateGraph(CmdState)
c.add_node("classify", classify_cmd)
for kind in ROUTES:
    c.add_node(kind, respond(kind))
c.add_node("fallback", lambda s: {"reply": "Sorry, I did not understand. Could you rephrase?"})
c.add_edge(START, "classify")                                        # no conditional edge needed
for kind in ROUTES + ["fallback"]:
    c.add_edge(kind, END)
cmd_app = c.compile()
out = cmd_app.invoke({"message": "What does error CH05 mean?"})
print("\n".join(out["log"]))
print("reply:", out["reply"])

## 11.3 The safety net: `recursion_limit`
A loop with a bug would run for ever. LangGraph stops every graph after a **recursion limit** of steps and raises an error. Build a loop that can **never** succeed (it always uses the loose prompt and never gives up) and watch the limit stop it:

In [ ]:
def classify_forever(state: CmdState) -> Command[Literal["act", "ask", "chat", "other", "classify"]]:
    raw = router_model.invoke([SystemMessage(LOOSE), HumanMessage(state["message"])]).content.strip().lower()
    label = next((r for r in ROUTES if raw.startswith(r)), "")
    return Command(update={"log": [f"tried: {raw[:30]!r}"]}, goto=label or "classify")      # no attempts counter: a bug

bug = StateGraph(CmdState)
bug.add_node("classify", classify_forever)
for kind in ROUTES:
    bug.add_node(kind, respond(kind))
bug.add_edge(START, "classify")
for kind in ROUTES:
    bug.add_edge(kind, END)
bug_app = bug.compile()
try:
    bug_app.invoke({"message": "Turn on the kitchen light"}, config={"recursion_limit": 6})
    print("it happened to find a valid label inside 6 steps")
    hit_limit = False
except GraphRecursionError as e:
    print("Stopped by the recursion limit:", str(e)[:100], "...")
    hit_limit = True

In [ ]:
check(hit_limit,
      "The runaway loop was stopped by the recursion limit instead of running all night. Always add your own attempts counter too: the limit is the last safety net, not the plan.",
      "the loose prompt should never give a valid label, so the limit should have stopped the graph.")

## 11.4 Fan-out and fan-in: three branches at the same time
A user says: *"Play a Hindi comedy for the kids on Netflix."* Three things can be done **independently**:
1. **search** the catalogue (a slow data lookup, simulated with a short delay);
2. **understand** the request with the small model (language, genre, audience, platform);
3. **check** the subscription (another slow lookup, simulated).

Three edges leave `START`, so the three nodes run **in the same step**. A **list** in `add_edge` makes `combine` **wait for all three**. Each branch **appends** to `findings`, so the state needs a **reducer**.

In [ ]:
CATALOG = json.load(open(DATA / "catalog.json", encoding="utf-8"))["items"]

class FanState(TypedDict, total=False):
    request: str
    findings: Annotated[list, operator.add]            # every branch adds one line
    reply: str

def search(state: FanState) -> dict:
    time.sleep(0.4)                                    # pretend this is a slow API
    hits = [i["title"] for i in CATALOG if i["language"] == "Hindi" and "Comedy" in i["genres"] and i.get("family_friendly")]
    return {"findings": [f"catalogue: {len(hits)} Hindi family comedies, for example {hits[:2]}"]}

def understand(state: FanState) -> dict:
    out = json_model.invoke([SystemMessage('Extract from the request: language, genre, audience, platform. Reply with one line of JSON only, like {"language": "...", "genre": "...", "audience": "...", "platform": "..."}.'),
                             HumanMessage(state["request"])]).content.strip()
    return {"findings": [f"request understood as: {out[:110]}"]}

def subscription(state: FanState) -> dict:
    time.sleep(0.4)                                    # another slow lookup
    return {"findings": ["subscription: the household has Netflix and Prime Video"]}

def combine(state: FanState) -> dict:
    facts = "\n".join(state["findings"])
    text = talk_model.invoke([SystemMessage("You are a TV assistant. Using ONLY these facts, write one short sentence for the user."),
                              HumanMessage(f"Request: {state['request']}\nFacts:\n{facts}")]).content.strip()
    return {"reply": text}

f = StateGraph(FanState)
for name, fn in [("search", search), ("understand", understand), ("subscription", subscription), ("combine", combine)]:
    f.add_node(name, fn)
f.add_edge(START, "search"); f.add_edge(START, "understand"); f.add_edge(START, "subscription")     # fan-out
f.add_edge(["search", "understand", "subscription"], "combine")                                      # fan-in: wait for all three
f.add_edge("combine", END)
fan_app = f.compile()
draw_graph(fan_app, "Fan-out: three branches at the same time, then combine")

In [ ]:
REQUEST = "Play a Hindi comedy for the kids on Netflix."
t0 = time.perf_counter()
out = fan_app.invoke({"request": REQUEST})
t_parallel = time.perf_counter() - t0
print("\n".join(out["findings"]))
print("\nreply:", out["reply"])
print(f"\nfan-out took {t_parallel:.1f} s")

**Read it:** `findings` holds **three lines, one per branch**, in whatever order the branches finished. The `combine` node wrote its sentence from all three. Now run the **same four nodes one after another** and compare the time:

In [ ]:
s = StateGraph(FanState)
for name, fn in [("search", search), ("understand", understand), ("subscription", subscription), ("combine", combine)]:
    s.add_node(name, fn)
s.add_edge(START, "search"); s.add_edge("search", "understand"); s.add_edge("understand", "subscription")
s.add_edge("subscription", "combine"); s.add_edge("combine", END)
seq_app = s.compile()
t0 = time.perf_counter()
seq_app.invoke({"request": REQUEST})
t_seq = time.perf_counter() - t0
print(f"one after another: {t_seq:.1f} s   |   fan-out: {t_parallel:.1f} s   |   saved: {t_seq - t_parallel:.1f} s")

In [ ]:
check(len(out["findings"]) == 3 and t_seq > t_parallel,
      f"Three branches, three findings, and the fan-out finished {t_seq - t_parallel:.1f} s sooner. The two slow lookups ran while the model was thinking.",
      "findings should hold three lines and the fan-out should be faster than the sequential chain.")

### Why a reducer?
Two branches writing the **same key** in the **same step** is ambiguous: which one wins? LangGraph refuses, unless the key has a **reducer** that says how to combine them. Remove the reducer and see:

In [ ]:
class NoReducer(TypedDict, total=False):
    request: str
    findings: list                                      # no reducer
    reply: str

n = StateGraph(NoReducer)
n.add_node("search", lambda state: search(state)); n.add_node("subscription", lambda state: subscription(state))   # plain wrappers: this state type has no reducer
n.add_edge(START, "search"); n.add_edge(START, "subscription")
n.add_edge("search", END); n.add_edge("subscription", END)
try:
    n.compile().invoke({"request": REQUEST})
    refused = False
except InvalidUpdateError as e:
    print("LangGraph refused:", str(e)[:140])
    refused = True
check(refused, "Without a reducer the graph stops with an error instead of silently losing one result. A reducer is your rule for combining parallel results.",
      "two branches writing findings without a reducer should raise InvalidUpdateError.")

## 11.5 Your turn: add a fourth branch
### 🔧 TODO 11 — Check the parental rating
Add a fourth parallel branch, `rating`. It should add one line to `findings`, for example *"rating: suitable for children"*. Wire it into the fan-out **and** the fan-in.

In [ ]:
def rating(state: FanState) -> dict:
    # TODO: return {"findings": ["rating: ..."]}
    return {"findings": []}

f4 = StateGraph(FanState)
for name, fn in [("search", search), ("understand", understand), ("subscription", subscription), ("rating", rating), ("combine", combine)]:
    f4.add_node(name, fn)
f4.add_edge(START, "search"); f4.add_edge(START, "understand"); f4.add_edge(START, "subscription")
# TODO: add the fan-out edge to "rating"
# TODO: change the fan-in so that combine waits for all FOUR branches
f4.add_edge("combine", END)
app4 = f4.compile()
out4 = app4.invoke({"request": REQUEST})
print("\n".join(out4["findings"]))

In [ ]:
check(len(out4["findings"]) == 4 and any(x.startswith("rating") for x in out4["findings"]),
      "Four branches, four findings. Adding parallel work is one edge out and one more name in the fan-in list.",
      "findings should have four lines including one that starts with 'rating'.")

## 11.6 Measure the router with its retry loop
Twelve labelled messages through the full router. Count how many needed a **retry** and how many ended in the **fallback**.

In [ ]:
LABELLED = [("Turn on the kitchen light", "act"), ("Set the bedroom AC to 21", "act"), ("Close the blinds in the study", "act"),
            ("What does error CH05 mean?", "ask"), ("How do I reset the filter alarm?", "ask"), ("Which rooms have blinds?", "ask"),
            ("Hello there", "chat"), ("Thanks, that was helpful", "chat"), ("Good night!", "chat"),
            ("Book me a table at an Italian restaurant", "other"), ("What is the capital of France?", "other"), ("Tell me a joke about cricket", "other")]
rows = []
for msg, want in LABELLED:
    o = router_app.invoke({"message": msg})
    rows.append({"message": msg, "wanted": want, "got": o["route"] or "fallback", "attempts": o["attempts"]})
rt = pd.DataFrame(rows)
rt["right"] = rt["wanted"] == rt["got"]
print(f"right: {rt['right'].mean():.0%}   needed a retry: {(rt['attempts'] > 1).sum()} of {len(rt)}   fallback: {(rt['got'] == 'fallback').sum()}")
rt

In [ ]:
check(rt["right"].mean() >= 0.5 and (rt["attempts"] > 1).any(),
      f"The router is right {rt['right'].mean():.0%} of the time and the retry loop rescued {(rt['attempts'] > 1).sum()} messages that would otherwise have been lost. Measure the router, because 'mostly works' is not a number.",
      "the router should be right at least half the time and the retry should have been used.")

## 11.7 What you proved
| You built | LangGraph idea | Why it matters |
|---|---|---|
| a router that retried | **conditional edge** pointing backwards (a loop) | never trust a label until you checked it |
| `log` that kept every attempt | **reducer** (`operator.add`) | parallel and repeated updates need a combining rule |
| `Command(update, goto)` | a node that writes the state and chooses the next node | decision and update in one place |
| the runaway loop | **`recursion_limit`** | the last safety net, not the plan |
| three branches then `combine` | **fan-out** and **fan-in** | independent work runs together |

## Check your understanding
1. Why did the first attempt in 11.1 fail on purpose, and what did the graph do about it?
2. What does the reducer `operator.add` do, and what happened in 11.4 when it was missing?
3. When would you prefer `Command(update=..., goto=...)` over a separate conditional edge?
4. Why does the fan-out finish sooner than the same nodes in a chain, and when would it **not**?

*✍️ Your answer:* 

## Next
Lab 12 gives the graph a memory: a **checkpointer** for the conversation, a **store** for facts about the person, and a **pause** for a human.